# Pipeline Completo: Predição de Inadimplência (AMEX)

Este notebook unifica todas as fases do projeto em sequência:

1. **Pipeline de Dados** — Engenharia temporal, agregação, merge, split e seleção de features
2. **Fase 1** — Provas de Conceito (Dimensionalidade e Balanceamento)
3. **Fase 2** — Campeonato Aberto (Benchmark dos 7 modelos)
4. **Fase 3** — Otimização Bayesiana (Optuna)
5. **Fase 4** — Meta-Classificadores (Ensembles)
6. **Fase 5** — Teste Final (Produção)
7. **Visualização** — Geração de gráficos

---

**Observação:** Este notebook serve como referência de execução e documentação. Para execução completa, é necessário que os dados brutos estejam disponíveis em `data/raw/parquet/`.

## 0. Setup e Imports Globais

In [ ]:
import sys
import os
import time
import logging
import gc
import json
import glob
import warnings
import numpy as np
import pandas as pd
import polars as pl
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.autolayout': True})

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-7s | %(message)s')
logger = logging.getLogger(__name__)

# Garantir que o diretório raiz do projeto esteja no path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Diretório do Projeto: {PROJECT_ROOT}')

In [ ]:
# Importações do projeto
from config import (
    RANDOM_SEED, RESULTS_DIR, RESULTS_BEST_MODELS,
    TRAIN_DATA_PATH, TEST_DATA_PATH, SELECTED_FEATURES_PATH,
    N_SPLITS, GPU_AVAILABLE, HYPERPARAMS, OPTUNA_GRIDS
)
from src.evaluation.amex_metric import amex_metric
from src.evaluation.metrics import evaluate_model
from src.models.registry import MODEL_REGISTRY

print(f'Seed: {RANDOM_SEED}')
print(f'GPU Disponível: {GPU_AVAILABLE}')
print(f'K-Folds: {N_SPLITS}')

---
## 1. Pipeline de Dados

O pipeline de preparação de dados transforma os dados brutos particionados em um dataset tabular pronto para modelagem. As etapas são:

1. **Engenharia Temporal (DuckDB):** Calcula `_diff1` (variação entre meses) para numéricos e `_changed` (flag de transição) para categóricos usando Window Functions.
2. **Agregação por Cliente (Polars):** Transforma a série temporal (~5.5M linhas) em uma única linha por cliente (~458K), calculando mean, std, min, max, last, trend_ratio, etc.
3. **Merge + Split Estratificado:** Une features com labels e divide 80/20 preservando proporção de classes.
4. **Feature Selection (LightGBM):** Reduz de ~3.265 colunas para 400 via importância Gain.

> **Nota:** Esta etapa requer os dados brutos em `data/raw/parquet/`. Caso já possua os dados processados, pule para a Seção 2.

In [ ]:
# Caminhos do Pipeline de Dados
caminho_input_glob = str(PROJECT_ROOT / 'data' / 'raw' / 'parquet' / 'train' / 'data_*.parquet')
caminho_labels_glob = str(PROJECT_ROOT / 'data' / 'raw' / 'parquet' / 'train_labels' / 'data_*.parquet')
caminho_output_dir = str(PROJECT_ROOT / 'data' / 'processed')
arquivo_intermediario = os.path.join(caminho_output_dir, 'temp_temporal.parquet')
arquivo_final = os.path.join(caminho_output_dir, 'dataset_final.parquet')

os.makedirs(caminho_output_dir, exist_ok=True)

# Verifica se os dados brutos existem
arquivos_encontrados = glob.glob(caminho_input_glob)
print(f'Arquivos de dados brutos encontrados: {len(arquivos_encontrados)}')

if not arquivos_encontrados:
    print('Dados brutos não encontrados. Pulando o Pipeline de Dados.')
    print('Para executar esta seção, posicione os arquivos Parquet em data/raw/parquet/train/')
    PIPELINE_DISPONIVEL = False
else:
    PIPELINE_DISPONIVEL = True

### 1.1 Pipeline: Engenharia Temporal com DuckDB

In [ ]:
if PIPELINE_DISPONIVEL:
    from app.pipeline.feature_engineering import EngenhariaTemporal

    logger.info('=== PIPELINE FASE 1: Engenharia Temporal (DuckDB) ===')
    conn = duckdb.connect(':memory:')

    try:
        schema_df = conn.execute(f"DESCRIBE SELECT * FROM read_parquet('{caminho_input_glob}')").df()
        colunas_originais = schema_df['column_name'].tolist()

        tabela_leitura = f"read_parquet('{caminho_input_glob}')"
        engenheiro = EngenhariaTemporal()
        sql_temporal = engenheiro.gerar_sql_temporal(tabela_origem=tabela_leitura, colunas_totais=colunas_originais)

        query_duckdb = f"""
        COPY (
            {sql_temporal}
        ) TO '{arquivo_intermediario}' (FORMAT PARQUET);
        """

        logger.info('Executando SQL de Engenharia Temporal...')
        conn.execute(query_duckdb)
        logger.info('Fase 1 do Pipeline (DuckDB) concluída!')
    finally:
        conn.close()
        gc.collect()
else:
    print('Pipeline Fase 1 ignorada (dados brutos indisponíveis).')

### 1.2 Pipeline: Agregação por Cliente com Polars

In [ ]:
if PIPELINE_DISPONIVEL:
    from app.pipeline.aggregation import AgregadorClientePolars

    logger.info('=== PIPELINE FASE 2: Agregação por Cliente (Polars) ===')
    agregador = AgregadorClientePolars()

    lazy_df = pl.scan_parquet(arquivo_intermediario)
    lazy_final = agregador.transformar(lazy_df)
    lazy_final.sink_parquet(arquivo_final)

    logger.info('Fase 2 do Pipeline (Polars) concluída!')

    # Limpeza
    if os.path.exists(arquivo_intermediario):
        os.remove(arquivo_intermediario)
        logger.info('Arquivo intermediário removido.')
else:
    print('Pipeline Fase 2 ignorada (dados brutos indisponíveis).')

### 1.3 Pipeline: Merge com Labels + Split Estratificado

In [ ]:
if PIPELINE_DISPONIVEL:
    from app.pipeline.merge_split import merge_and_split

    logger.info('=== PIPELINE FASE 3: Merge e Split Estratificado ===')
    merge_and_split(arquivo_final, caminho_labels_glob, caminho_output_dir)
    logger.info('Fase 3 do Pipeline (Merge/Split) concluída!')
else:
    print('Pipeline Fase 3 ignorada (dados brutos indisponíveis).')

### 1.4 Pipeline: Feature Selection (LightGBM Gain)

In [ ]:
if PIPELINE_DISPONIVEL:
    from app.pipeline.feature_selection import SelecionadorFeaturesAMEX

    logger.info('=== PIPELINE FASE 4: Feature Selection ===')
    caminho_treino_split = str(PROJECT_ROOT / 'data' / 'processed' / 'merge_split' / 'train_80.parquet')
    caminho_saida_selecao = str(PROJECT_ROOT / 'data' / 'processed' / 'selection' / 'train_80_selected.parquet')
    caminho_lista_features = str(PROJECT_ROOT / 'data' / 'processed' / 'selection' / 'selected_features_list.txt')

    os.makedirs(os.path.dirname(caminho_saida_selecao), exist_ok=True)

    df_train_80 = pl.read_parquet(caminho_treino_split)
    selecionador = SelecionadorFeaturesAMEX()
    df_train_selected = selecionador.selecionar(df_train_80)

    df_train_selected.write_parquet(caminho_saida_selecao)

    with open(caminho_lista_features, 'w') as f:
        for col in df_train_selected.columns:
            f.write(f'{col}\n')

    logger.info(f'Feature Selection concluída! {len(df_train_selected.columns)} features selecionadas.')
else:
    print('Pipeline Fase 4 ignorada (dados brutos indisponíveis).')

---
## 2. Carregamento dos Dados Processados

A partir daqui, trabalhamos com os dados já processados pelo pipeline (`train_80.parquet` + lista de 400 features selecionadas).

In [ ]:
def load_and_prepare_data():
    """Carrega a base de treino processada e aplica o Feature Selection."""
    logger.info('Carregando base de treino via Polars...')
    df_pd = pl.scan_parquet(TRAIN_DATA_PATH).collect().to_pandas()

    # Remove colunas de texto
    cols_to_drop = [col for col in ['customer_ID', 'S_2'] if col in df_pd.columns]
    if cols_to_drop:
        df_pd = df_pd.drop(columns=cols_to_drop)
    object_cols = df_pd.select_dtypes(include=['object', 'string', 'category']).columns
    if len(object_cols) > 0:
        df_pd = df_pd.drop(columns=object_cols)

    # Separa alvo
    y = df_pd['target'].astype('int8')
    X_full = df_pd.drop(columns=['target']).astype('float32')
    del df_pd
    gc.collect()

    # Aplica Feature Selection
    with open(SELECTED_FEATURES_PATH, 'r') as f:
        selected_cols = [line.strip() for line in f.readlines()]
    if 'target' in selected_cols:
        selected_cols.remove('target')
    selected_cols = [col for col in selected_cols if col in X_full.columns]

    X_reduced = X_full[selected_cols]
    del X_full
    gc.collect()

    return X_reduced, y


# Carrega os dados
X, y = load_and_prepare_data()
print(f'Shape dos dados: X={X.shape}, y={y.shape}')
print(f'Distribuição de classes: {y.value_counts().to_dict()}')

---
## 3. Fase 1: Provas de Conceito

Dois experimentos fundamentais:
1. **Dimensionalidade:** Comprova a eficácia do Feature Selection (Base Completa vs Base Enxuta)
2. **Balanceamento:** Comprova a superioridade do Balanceamento Algorítmico vs Undersampling

### 3.1 Experimento 1: Dimensionalidade


In [ ]:
# (Nota: Para este experimento, carregamos a base completa também)

logger.info('=== FASE 1 - Experimento 1: Dimensionalidade ===')

# Carrega a base completa para comparação
df_full_pl = pl.scan_parquet(TRAIN_DATA_PATH).collect().to_pandas()
cols_to_drop = [col for col in ['customer_ID', 'S_2'] if col in df_full_pl.columns]
if cols_to_drop:
    df_full_pl = df_full_pl.drop(columns=cols_to_drop)
object_cols = df_full_pl.select_dtypes(include=['object', 'string', 'category']).columns
if len(object_cols) > 0:
    df_full_pl = df_full_pl.drop(columns=object_cols)

y_poc = df_full_pl['target'].astype('int8')
X_full = df_full_pl.drop(columns=['target']).astype('float32')
del df_full_pl
gc.collect()

# Base enxuta (já carregada)
X_reduced = X

results_dim = []
datasets = {'Completa': X_full, 'Enxuta (400 features)': X_reduced}

xgb_params = {'scale_pos_weight': 3, 'n_estimators': 200, 'max_depth': 6, 'random_state': RANDOM_SEED}
if GPU_AVAILABLE:
    xgb_params['tree_method'] = 'hist'
    xgb_params['device'] = 'cuda'

models_poc = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=500, random_state=RANDOM_SEED))
    ]),
    'XGBoost': XGBClassifier(**xgb_params)
}

for db_name, X_data in datasets.items():
    X_train, X_val, y_train, y_val = train_test_split(X_data, y_poc, test_size=0.2, stratify=y_poc, random_state=RANDOM_SEED)
    gc.collect()

    for model_name, model in models_poc.items():
        start = time.time()
        model.fit(X_train, y_train)
        t = time.time() - start

        preds = model.predict_proba(X_val)[:, 1]
        metrics = evaluate_model(y_val, preds)

        results_dim.append({
            'Base': db_name, 'Modelo': model_name,
            'Tempo (s)': round(t, 2),
            'AMEX Score': metrics['AMEX_Score'],
            'ROC AUC': metrics['ROC_AUC']
        })
        print(f'[{model_name}] {db_name} -> AMEX: {metrics["AMEX_Score"]:.4f} | Tempo: {t:.1f}s')
        gc.collect()

df_dim = pd.DataFrame(results_dim)
del X_full
gc.collect()

df_dim

### 3.2 Experimento 2: Estratégias de Balanceamento

In [ ]:
logger.info('=== FASE 1 - Experimento 2: Balanceamento ===')

X_train_bal, X_val_bal, y_train_bal, y_val_bal = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

strategies = ['Sem Balanceamento', 'Undersampling (Físico)', 'Algorítmico (Cost-Sensitive)']
results_bal = []

for strategy in strategies:
    X_train_run, y_train_run = X_train_bal, y_train_bal
    lr_kwargs = {'max_iter': 500, 'random_state': RANDOM_SEED}
    xgb_kwargs = {'n_estimators': 200, 'max_depth': 6, 'random_state': RANDOM_SEED}
    if GPU_AVAILABLE:
        xgb_kwargs['tree_method'] = 'hist'
        xgb_kwargs['device'] = 'cuda'

    if strategy == 'Undersampling (Físico)':
        sampler = RandomUnderSampler(random_state=RANDOM_SEED)
        X_train_run, y_train_run = sampler.fit_resample(X_train_bal, y_train_bal)
    elif strategy == 'Algorítmico (Cost-Sensitive)':
        lr_kwargs['class_weight'] = 'balanced'
        xgb_kwargs['scale_pos_weight'] = 3

    models_bal = {
        'Logistic Regression': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('classifier', LogisticRegression(**lr_kwargs))
        ]),
        'XGBoost': XGBClassifier(**xgb_kwargs)
    }

    for model_name, model in models_bal.items():
        start = time.time()
        model.fit(X_train_run, y_train_run)
        t = time.time() - start

        preds = model.predict_proba(X_val_bal)[:, 1]
        metrics = evaluate_model(y_val_bal, preds)

        results_bal.append({
            'Estratégia': strategy, 'Modelo': model_name,
            'Tempo (s)': round(t, 2),
            'AMEX Score': metrics['AMEX_Score'],
            'Recall': metrics['Recall']
        })
        print(f'[{model_name}] {strategy} -> AMEX: {metrics["AMEX_Score"]:.4f} | Recall: {metrics["Recall"]:.4f}')
        gc.collect()

df_bal = pd.DataFrame(results_bal)
df_bal

### 3.3 Salvar resultados da Fase 1

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df_dim.to_csv(RESULTS_DIR / 'poc_01_dimensionalidade.csv', index=False)
df_bal.to_csv(RESULTS_DIR / 'poc_02_balanceamento.csv', index=False)
print('Resultados da Fase 1 salvos em results/')

---
## 4. Fase 2: Campeonato Aberto (Benchmark dos 7 Modelos)

Avalia os 7 algoritmos base utilizando validação cruzada estratificada (5-Fold OOF).
Gera o ranking pelo AMEX Score para selecionar o Top 3 para otimização.

In [ ]:
BASE_MODELS = [
    'Logistic Regression', 'KNN', 'ANN (MLP)',
    'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost'
]

def wrap_model_if_needed(model_name, model_instance):
    """Modelos clássicos precisam de imputação; modelos de árvore lidam com NaN nativamente."""
    modern = ['XGBoost', 'LightGBM', 'CatBoost']
    if model_name in modern:
        return model_instance
    return Pipeline([('imputer', SimpleImputer(strategy='median')), ('classifier', model_instance)])


logger.info('=== FASE 2: CAMPEONATO ABERTO ===')
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
results_phase2 = []

for model_name in BASE_MODELS:
    if model_name not in MODEL_REGISTRY:
        print(f'[SKIP] {model_name} não encontrado no Registry.')
        continue

    print(f'\n-> Treinando: {model_name}')
    start = time.time()

    raw_model = MODEL_REGISTRY[model_name]()
    model = wrap_model_if_needed(model_name, raw_model)

    oof_preds = np.zeros(len(X))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train = pd.DataFrame(X.iloc[train_idx].values, columns=X.columns)
        y_train = pd.Series(y.iloc[train_idx].values)
        X_val = pd.DataFrame(X.iloc[val_idx].values, columns=X.columns)

        model.fit(X_train, y_train)
        preds = model.predict_proba(X_val)
        preds_pos = preds[:, 1] if len(preds.shape) > 1 else preds
        oof_preds[val_idx] = preds_pos

        fold_metric = evaluate_model(y.iloc[val_idx], preds_pos)
        fold_scores.append(fold_metric['AMEX_Score'])
        print(f'Fold {fold+1}/{N_SPLITS} | AMEX: {fold_metric["AMEX_Score"]:.4f}')
        gc.collect()

    total_time = time.time() - start
    global_metrics = evaluate_model(y, oof_preds)

    results_phase2.append({
        'Modelo': model_name,
        'AMEX Score (OOF)': global_metrics['AMEX_Score'],
        'ROC AUC': global_metrics['ROC_AUC'],
        'AUPRC': global_metrics['AUPRC'],
        'F1': global_metrics['F1_Score'],
        'Tempo Total (s)': round(total_time, 1),
        'Std (Folds)': round(np.std(fold_scores), 4)
    })
    print(f'GLOBAL -> AMEX: {global_metrics["AMEX_Score"]:.4f} | Tempo: {total_time:.1f}s')

df_phase2 = pd.DataFrame(results_phase2).sort_values(by='AMEX Score (OOF)', ascending=False).reset_index(drop=True)
df_phase2

### 4.1 Salvar ranking da Fase 2

In [ ]:
df_phase2.to_csv(RESULTS_DIR / 'phase2_benchmark_ranking.csv', index=False)
print('Ranking da Fase 2 salvo!')
print(f'\nTop 3 para a Fase 3: {df_phase2["Modelo"].head(3).tolist()}')

---
## 5. Fase 3: Otimização Bayesiana (Optuna)

Aplica o Optuna nos 3 modelos campeões da Fase 2 (XGBoost, LightGBM, CatBoost).
Maximiza o AMEX Score via busca bayesiana com 50-100 trials por modelo.

> **Atenção:** Esta fase pode levar várias horas em CPU. Os resultados já salvos estão em `results/best_models/optuna_best_params.json`.

In [ ]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 100 if GPU_AVAILABLE else 50
top3_models = ['XGBoost', 'LightGBM', 'CatBoost']

def get_model_instance_optuna(model_name, trial_params):
    """Instancia o modelo com os parâmetros sugeridos pelo Optuna."""
    if model_name == 'LightGBM':
        return LGBMClassifier(
            **trial_params, is_unbalance=True, random_state=RANDOM_SEED,
            n_jobs=-1 if GPU_AVAILABLE else 1,
            device='gpu' if GPU_AVAILABLE else 'cpu', verbose=-1
        )
    elif model_name == 'XGBoost':
        return XGBClassifier(
            **trial_params, scale_pos_weight=3, eval_metric='logloss',
            random_state=RANDOM_SEED, n_jobs=-1,
            tree_method='gpu_hist' if GPU_AVAILABLE else 'hist',
            device='cuda' if GPU_AVAILABLE else 'cpu'
        )
    elif model_name == 'CatBoost':
        return CatBoostClassifier(
            **trial_params, auto_class_weights='Balanced',
            random_seed=RANDOM_SEED, verbose=0,
            task_type='GPU' if GPU_AVAILABLE else 'CPU'
        )


def objective(trial, model_name, X_opt, y_opt, skf_opt):
    """Função Objetivo: treina K-Fold e retorna AMEX Score."""
    grid = OPTUNA_GRIDS[model_name]
    trial_params = {}

    for param_name, param_config in grid.items():
        param_type = param_config[0]
        if param_type == 'int':
            trial_params[param_name] = trial.suggest_int(param_name, param_config[1], param_config[2])
        elif param_type == 'float':
            log = len(param_config) == 4 and param_config[3] == 'log'
            trial_params[param_name] = trial.suggest_float(param_name, param_config[1], param_config[2], log=log)

    # Proteção matemática do LightGBM
    if model_name == 'LightGBM' and 'max_depth' in trial_params and 'num_leaves' in trial_params:
        max_allowed = (2 ** trial_params['max_depth']) - 1
        trial_params['num_leaves'] = min(trial_params['num_leaves'], max_allowed)

    model = get_model_instance_optuna(model_name, trial_params)
    oof_preds = np.zeros(len(X_opt))

    for train_idx, val_idx in skf_opt.split(X_opt, y_opt):
        X_train = pd.DataFrame(X_opt.iloc[train_idx].values, columns=X_opt.columns)
        y_train = pd.Series(y_opt.iloc[train_idx].values)
        X_val = pd.DataFrame(X_opt.iloc[val_idx].values, columns=X_opt.columns)

        model.fit(X_train, y_train)
        preds = model.predict_proba(X_val)
        oof_preds[val_idx] = preds[:, 1] if len(preds.shape) > 1 else preds

        del X_train, y_train, X_val
        gc.collect()

    return evaluate_model(y_opt, oof_preds)['AMEX_Score']


print(f'Configuração: {N_TRIALS} trials por modelo | GPU: {GPU_AVAILABLE}')

### 5.1 Execução da Otimização

In [ ]:
logger.info('=== FASE 3: OTIMIZAÇÃO BAYESIANA (OPTUNA) ===')

RESULTS_BEST_MODELS.mkdir(parents=True, exist_ok=True)
skf_opt = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
best_params_all = {}

for model_name in top3_models:
    print(f'\n-> Otimizando: {model_name} ({N_TRIALS} trials)')
    start = time.time()

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_SEED))

    def save_checkpoint(study, trial):
        best_params_all[model_name] = {'best_amex_score': study.best_value, 'params': study.best_params}
        with open(RESULTS_BEST_MODELS / 'optuna_best_params.json', 'w') as f:
            json.dump(best_params_all, f, indent=4)

    study.optimize(
        lambda trial: objective(trial, model_name, X, y, skf_opt),
        n_trials=N_TRIALS, n_jobs=1, callbacks=[save_checkpoint]
    )

    total_time = time.time() - start
    best_params_all[model_name] = {'best_amex_score': study.best_value, 'params': study.best_params}

    print(f'Melhor AMEX: {study.best_value:.4f} | Tempo: {total_time:.1f}s')
    print(f'Params: {study.best_params}')

# Salva JSON final
with open(RESULTS_BEST_MODELS / 'optuna_best_params.json', 'w') as f:
    json.dump(best_params_all, f, indent=4)

print('\nFase 3 concluída! Parâmetros salvos em results_best_models/optuna_best_params.json')

---
## 6. Fase 4: Meta-Classificadores (Ensembles)

Combina os 3 modelos otimizados via:
1. **Soft Voting** — Média simples das probabilidades
2. **Stacking** — Regressão Logística sobre predições OOF
3. **Blending** — Meta-modelo com holdout fixo de 20%

In [ ]:
# Carrega os hiperparâmetros otimizados
logger.info('=== FASE 4: META-CLASSIFICADORES ===')

json_path = RESULTS_BEST_MODELS / 'optuna_best_params.json'
with open(json_path, 'r') as f:
    best_params = json.load(f)

# Instancia modelos otimizados
xgb_model = XGBClassifier(
    **best_params['XGBoost']['params'], scale_pos_weight=3, eval_metric='logloss',
    random_state=RANDOM_SEED, n_jobs=-1,
    tree_method='gpu_hist' if GPU_AVAILABLE else 'hist',
    device='cuda' if GPU_AVAILABLE else 'cpu'
)
lgb_model = LGBMClassifier(
    **best_params['LightGBM']['params'], is_unbalance=True,
    random_state=RANDOM_SEED, n_jobs=-1 if GPU_AVAILABLE else 1,
    device='gpu' if GPU_AVAILABLE else 'cpu', verbose=-1
)
cat_model = CatBoostClassifier(
    **best_params['CatBoost']['params'], auto_class_weights='Balanced',
    random_seed=RANDOM_SEED, verbose=0,
    task_type='GPU' if GPU_AVAILABLE else 'CPU'
)

print('Modelos otimizados instanciados:')
for name in ['XGBoost', 'LightGBM', 'CatBoost']:
    print(f'{name}: AMEX Score (Fase 3) = {best_params[name]["best_amex_score"]:.4f}')

### 6.1 Geração da Matriz Base OOF

In [ ]:
skf_ens = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

print('Gerando predições OOF para os 3 modelos...')
start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf_ens.split(X, y)):
    print(f'Fold {fold+1}/{N_SPLITS}...')
    X_train = pd.DataFrame(X.iloc[train_idx].values, columns=X.columns)
    y_train = pd.Series(y.iloc[train_idx].values)
    X_val = pd.DataFrame(X.iloc[val_idx].values, columns=X.columns)

    xgb_model.fit(X_train, y_train)
    lgb_model.fit(X_train, y_train)
    cat_model.fit(X_train, y_train)

    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]

    del X_train, y_train, X_val
    gc.collect()

print(f'Matriz OOF concluída em {time.time()-start:.1f}s')
print(f'XGBoost OOF AMEX: {evaluate_model(y, oof_xgb)["AMEX_Score"]:.4f}')
print(f'LightGBM OOF AMEX: {evaluate_model(y, oof_lgb)["AMEX_Score"]:.4f}')
print(f'CatBoost OOF AMEX: {evaluate_model(y, oof_cat)["AMEX_Score"]:.4f}')

### 6.2 Execução dos Meta-Classificadores

In [ ]:
# Soft Voting Classifier
print('--- Soft Voting Classifier ---')
oof_voting = (oof_xgb + oof_lgb + oof_cat) / 3
voting_metrics = evaluate_model(y, oof_voting)
print(f'AMEX Score (Voting): {voting_metrics["AMEX_Score"]:.4f}')

# Stacking Classifier
print('\n--- Stacking Classifier ---')
X_meta = pd.DataFrame({'xgb': oof_xgb, 'lgb': oof_lgb, 'cat': oof_cat})
meta_model = LogisticRegression(random_state=RANDOM_SEED)
oof_stacking = cross_val_predict(meta_model, X_meta, y, cv=skf_ens, method='predict_proba')[:, 1]
stacking_metrics = evaluate_model(y, oof_stacking)
print(f'AMEX Score (Stacking): {stacking_metrics["AMEX_Score"]:.4f}')

# Blending Classifier
print('\n--- Blending Classifier ---')
X_blend_train, X_blend_val, y_blend_train, y_blend_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
X_blend_train = pd.DataFrame(X_blend_train.values, columns=X.columns)
X_blend_val = pd.DataFrame(X_blend_val.values, columns=X.columns)

xgb_model.fit(X_blend_train, y_blend_train)
lgb_model.fit(X_blend_train, y_blend_train)
cat_model.fit(X_blend_train, y_blend_train)

blend_xgb = xgb_model.predict_proba(X_blend_val)[:, 1]
blend_lgb = lgb_model.predict_proba(X_blend_val)[:, 1]
blend_cat = cat_model.predict_proba(X_blend_val)[:, 1]

X_meta_blend = pd.DataFrame({'xgb': blend_xgb, 'lgb': blend_lgb, 'cat': blend_cat})
meta_model.fit(X_meta_blend, y_blend_val)
preds_blending = meta_model.predict_proba(X_meta_blend)[:, 1]
blending_metrics = evaluate_model(y_blend_val, preds_blending)
print(f'AMEX Score (Blending): {blending_metrics["AMEX_Score"]:.4f}')

### 6.3 Salvar resultados da Fase 4

In [ ]:
# Ranking dos Ensembles
results_ensembles = [
    {'Modelo': 'Voting Classifier', 'AMEX Score': voting_metrics['AMEX_Score'], 'ROC AUC': voting_metrics['ROC_AUC']},
    {'Modelo': 'Stacking Classifier', 'AMEX Score': stacking_metrics['AMEX_Score'], 'ROC AUC': stacking_metrics['ROC_AUC']},
    {'Modelo': 'Blending Classifier', 'AMEX Score': blending_metrics['AMEX_Score'], 'ROC AUC': blending_metrics['ROC_AUC']},
]

df_ensembles = pd.DataFrame(results_ensembles).sort_values(by='AMEX Score', ascending=False).reset_index(drop=True)
df_ensembles.to_csv(RESULTS_DIR / 'phase4_ensembles_ranking.csv', index=False)

print('Ranking dos Meta-Classificadores:')
df_ensembles

---
## 7. Fase 5: Teste Final (Produção)

Treina os modelos em 100% da base de treino e avalia na base de teste isolada (20%, 91.783 clientes) que permaneceu intocada desde o início do projeto.

In [ ]:
logger.info('=== FASE 5: TESTE FINAL (PRODUÇÃO) ===')

# Carrega base de teste isolada
def load_dataset(path, selected_features=None):
    df_pd = pl.scan_parquet(path).collect().to_pandas()
    cols_to_drop = [col for col in ['customer_ID', 'S_2'] if col in df_pd.columns]
    if cols_to_drop:
        df_pd = df_pd.drop(columns=cols_to_drop)
    object_cols = df_pd.select_dtypes(include=['object', 'string', 'category']).columns
    if len(object_cols) > 0:
        df_pd = df_pd.drop(columns=object_cols)
    y_out = df_pd['target'].astype('int8')
    X_out = df_pd.drop(columns=['target']).astype('float32')
    del df_pd
    gc.collect()
    if selected_features is not None:
        valid_cols = [col for col in selected_features if col in X_out.columns]
        X_out = X_out[valid_cols]
    return X_out, y_out

# Lista de features selecionadas
with open(SELECTED_FEATURES_PATH, 'r') as f:
    features = [line.strip() for line in f.readlines()]
if 'target' in features:
    features.remove('target')

X_train_final, y_train_final = load_dataset(TRAIN_DATA_PATH, features)
X_test_final, y_test_final = load_dataset(TEST_DATA_PATH, features)

print(f'Treino: {X_train_final.shape} | Teste: {X_test_final.shape}')

# Verifica alinhamento
assert list(X_train_final.columns) == list(X_test_final.columns), 'Erro: colunas não coincidem!'

### 7.1 Reinstancia modelos com params otimizados e treina em 100%

In [ ]:
xgb_final = XGBClassifier(
    **best_params['XGBoost']['params'], scale_pos_weight=3, eval_metric='logloss',
    random_state=RANDOM_SEED, n_jobs=-1,
    tree_method='gpu_hist' if GPU_AVAILABLE else 'hist',
    device='cuda' if GPU_AVAILABLE else 'cpu'
)
lgb_final = LGBMClassifier(
    **best_params['LightGBM']['params'], is_unbalance=True,
    random_state=RANDOM_SEED, n_jobs=-1 if GPU_AVAILABLE else 1,
    device='gpu' if GPU_AVAILABLE else 'cpu', verbose=-1
)
cat_final = CatBoostClassifier(
    **best_params['CatBoost']['params'], auto_class_weights='Balanced',
    random_seed=RANDOM_SEED, verbose=0,
    task_type='GPU' if GPU_AVAILABLE else 'CPU'
)

print('Treinando modelos definitivos com 100% da base de treino...')
start = time.time()

xgb_final.fit(X_train_final, y_train_final)
print('--> XGBoost treinado.')

lgb_final.fit(X_train_final, y_train_final)
print('--> LightGBM treinado.')

cat_final.fit(X_train_final, y_train_final)
print('--> CatBoost treinado.')

print(f'Treinamento concluído em {time.time()-start:.1f}s')

# Libera memória
del X_train_final, y_train_final
gc.collect()

### 7.2 Predições no Teste (Voting Classifier)

In [ ]:
pred_xgb = xgb_final.predict_proba(X_test_final)[:, 1]
pred_lgb = lgb_final.predict_proba(X_test_final)[:, 1]
pred_cat = cat_final.predict_proba(X_test_final)[:, 1]

pred_voting_final = (pred_xgb + pred_lgb + pred_cat) / 3

# Métricas Finais
final_metrics = evaluate_model(y_test_final, pred_voting_final)

print('=' * 50)
print('  RESULTADOS DO TESTE FINAL (PRODUÇÃO)')
print('=' * 50)
print(f'AMEX Score    : {final_metrics["AMEX_Score"]:.4f}')
print(f'ROC AUC       : {final_metrics["ROC_AUC"]:.4f}')
print(f'AUPRC         : {final_metrics["AUPRC"]:.4f}')
print(f'F1-Score      : {final_metrics["F1_Score"]:.4f}')
print(f'Recall        : {final_metrics["Recall"]:.4f}')
print(f'Precision     : {final_metrics["Precision"]:.4f}')
print('=' * 50)

# Salva resultado final
df_final = pd.DataFrame([final_metrics])
df_final.insert(0, 'Modelo', 'Voting Classifier (Final)')
df_final.to_csv(RESULTS_DIR / 'phase5_final_test.csv', index=False)
print('\nResultado salvo em results/phase5_final_test.csv')

---
## 8. Visualização de Resultados

Gera os gráficos acadêmicos para o artigo científico.

### 8.1 Gráfico 1: Ranking da Fase 2

In [ ]:
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df_ranking = pd.read_csv(RESULTS_DIR / 'phase2_benchmark_ranking.csv').sort_values(by='AMEX Score (OOF)', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_ranking['Modelo'], df_ranking['AMEX Score (OOF)'], color=sns.color_palette('viridis', len(df_ranking)))
ax.set_title('Ranking de Baseline - Fase 2 (AMEX Score)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('AMEX Score Global', fontsize=14)
ax.set_xlim(0.5, 0.82)
for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.4f}', va='center', ha='left', fontsize=11, fontweight='bold')
plt.savefig(PLOTS_DIR / '01_phase2_ranking.png', dpi=300, bbox_inches='tight')
plt.show()

### 8.2 Gráfico 2: Evolução do AMEX Score

In [ ]:
f2_score = pd.read_csv(RESULTS_DIR / 'phase2_benchmark_ranking.csv')['AMEX Score (OOF)'].max()
f4_score = voting_metrics['AMEX_Score'] if 'voting_metrics' in dir() else 0.7920
f5_score = final_metrics['AMEX_Score'] if 'final_metrics' in dir() else 0.7931

fases = ['Fase 2\n(Melhor Baseline)', 'Fase 4\n(Voting OOF)', 'Fase 5\n(Teste Final)']
scores = [f2_score, f4_score, f5_score]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(fases, scores, color=['#A9A9A9', '#4682B4', '#2E8B57'], width=0.5)
ax.set_title('Evolução Preditiva do AMEX Score', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('AMEX Score', fontsize=14)
ax.set_ylim(0.7800, 0.7960)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{bar.get_height():.4f}', va='bottom', ha='center', fontsize=12, fontweight='bold')
plt.savefig(PLOTS_DIR / '02_amex_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

### 8.3 Gráfico 3: Matriz de Confusão (Fase 5)

In [ ]:
cm = np.array(final_metrics['Confusion_Matrix'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predito: Adimplente', 'Predito: Inadimplente'],
            yticklabels=['Real: Adimplente', 'Real: Inadimplente'])
ax.set_title('Matriz de Confusão - Teste Final', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Predição', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
plt.savefig(PLOTS_DIR / '03_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\nMatriz de Confusão ({cm.sum()} clientes):')
print(f'Verdadeiros Negativos: {cm[0,0]:,}')
print(f'Verdadeiros Positivos: {cm[1,1]:,}')
print(f'Falsos Positivos:      {cm[0,1]:,}')
print(f'Falsos Negativos:      {cm[1,0]:,}')

---
## 9. Resumo Final

| Fase | Descrição | Resultado Principal |
|------|-----------|--------------------|
| Pipeline | Eng. Temporal + Agregação + Seleção | 5.5M linhas → 458K clientes × 400 features |
| Fase 1 | Provas de Conceito | Feature Selection: +score, -tempo; Algorítmico > Undersampling |
| Fase 2 | Benchmark (7 modelos) | Top 3: XGBoost (0.7872), LightGBM (0.7871), CatBoost (0.7858) |
| Fase 3 | Otimização Optuna | LightGBM (0.7910), XGBoost (0.7900), CatBoost (0.7893) |
| Fase 4 | Ensembles | Voting (0.7920) > Stacking (0.7918) |
| Fase 5 | Teste Final | **Voting Classifier: AMEX 0.7931, ROC AUC 0.9618, Recall 0.9197** |